In [1]:
# reasoning_analysis.py
# -------------------------------------------------------------
# Analyze reasoning lines by MODEL × OPTION (A/B/C) and compare
# results across LLMs for each option.
#
# Outputs:
#   - reasoning_metrics_by_model_option.csv
#   - reasoning_counts_by_model_option.csv
#   - comparisons/
#       option_A_comparison.csv
#       option_B_comparison.csv
#       option_C_comparison.csv
#       rankings_option_<A|B|C>_<metric>.csv
#   - (optional if --plots) plots/*.png
#
# CLI examples:
#   python reasoning_analysis.py \
#     --file "gpt4o=/path/gpt4o_test.csv" \
#     --file "llama3.1=/path/llama3.1_test.csv" \
#     --output-dir ./out --plots
#
# Or call from Python:
#   from reasoning_analysis import run_analysis, compare_all_options
#   summary, counts = run_analysis({...}, choice_col="choice",
#                                  reasoning_cols=["reasoning","explanation"],
#                                  output_dir=".")
#   compare_all_options(summary, output_dir=".")
# -------------------------------------------------------------

import os
import re
import sys
import json
import math
import argparse
from typing import Dict, List, Optional, Tuple, Sequence
from collections import defaultdict

import pandas as pd

# -----------------------------
# Lexicons (editable)
# -----------------------------
DEFAULT_LEXICONS = {
    "hedges": [
        "maybe","perhaps","possibly","might","could","seems","appears","likely",
        "generally","typically","roughly","approximately","suggests","arguably",
        "often","sometimes","usually","tend","tends","potentially","somewhat"
    ],
    "confidence": [
        "must","should","clearly","certainly","definitely","undoubtedly","always",
        "never","ensure","guarantee","best","optimal","required","necessary"
    ],
    "justification_markers": [
        "because","since","as ","as it","therefore","thus","so that","so ",
        "hence","in order to","given that","due to"
    ],
    "moral": [
        "ethical","ethics","moral","duty","obligation","responsibility","rights",
        "fair","fairness","justice","harm","benefit","values","integrity",
        "honesty","respect","consent","transparency","accountability"
    ],
    "relational": [
        "relationship","collaborate","collaboration","cooperate","cooperation",
        "negotiat","compromise","conflict","harmony","trust","empathy","empath",
        "advisor","colleague","team","stakeholder","patient","doctor","manager",
        "supervisor","family","friend","partner","discuss","dialogue","listen"
    ],
    "epistemic": [
        "evidence","data","facts","accuracy","accurate","reliable","source",
        "uncertain","uncertainty","verify","validate","check","expert","authority",
        "research","experiment","analysis","reason","proof","knowledge"
    ],
    # Minimal sentiment lexicon (extend as needed)
    "positive": [
        "good","better","best","benefit","helpful","improve","positive","support",
        "clear","confident","appropriate","effective","responsible","fair","safe"
    ],
    "negative": [
        "bad","worse","worst","risk","harm","problem","negative","unsafe",
        "unclear","conflict","bias","wrong","fail","failure","costly","error"
    ]
}

# -----------------------------
# Text utilities
# -----------------------------
_TOKEN_RE = re.compile(r"[A-Za-z']+")
_SENT_RE = re.compile(r"[.!?]+")

def tokenize(text: str) -> List[str]:
    return _TOKEN_RE.findall(text.lower()) if isinstance(text, str) else []

def count_sentences(text: str) -> int:
    if not isinstance(text, str) or not text.strip():
        return 0
    return max(1, len(_SENT_RE.findall(text)))  # at least 1 if non-empty

def contains_any(text: str, terms: List[str]) -> bool:
    if not isinstance(text, str):
        return False
    tl = text.lower()
    return any(term in tl for term in terms)

def count_terms(tokens: List[str], terms: List[str]) -> int:
    terms_set = set(t.lower() for t in terms)
    return sum(1 for t in tokens if t in terms_set)

def rate_per_100_words(count: int, total_words: int) -> float:
    if total_words <= 0:
        return 0.0
    return 100.0 * count / total_words

def rate_per_1000_words(count: int, total_words: int) -> float:
    if total_words <= 0:
        return 0.0
    return 1000.0 * count / total_words

def simple_sentiment(tokens: List[str], pos: List[str], neg: List[str]) -> float:
    """Return a simple sentiment score in [-1, 1] using tiny lexicons."""
    if not tokens:
        return 0.0
    pos_set, neg_set = set(pos), set(neg)
    p = sum(1 for t in tokens if t in pos_set)
    n = sum(1 for t in tokens if t in neg_set)
    return 0.0 if (p + n) == 0 else (p - n) / (p + n)

# -----------------------------
# Core analysis per group
# -----------------------------
def analyze_reasonings(reasonings: List[str], lex=DEFAULT_LEXICONS) -> dict:
    """Compute metrics over a list of reasoning strings."""
    total_words = 0
    total_chars = 0
    total_sents = 0
    total_tokens = 0
    total_unique = 0

    hedge_hits = 0
    conf_hits = 0
    justi_hits = 0  # documents containing any justification marker

    moral_hits = 0
    relational_hits = 0
    epistemic_hits = 0

    sentiment_scores = []

    for r in reasonings:
        r = r if isinstance(r, str) else ""
        toks = tokenize(r)
        total_words += len(toks)
        total_chars += len(r)
        total_sents += count_sentences(r)
        total_tokens += len(toks)
        total_unique += len(set(toks))

        hedge_hits += count_terms(toks, lex["hedges"])
        conf_hits  += count_terms(toks, lex["confidence"])

        if contains_any(r, lex["justification_markers"]):
            justi_hits += 1

        moral_hits      += count_terms(toks, lex["moral"])
        relational_hits += count_terms(toks, lex["relational"])
        epistemic_hits  += count_terms(toks, lex["epistemic"])

        sentiment_scores.append(simple_sentiment(toks, lex["positive"], lex["negative"]))

    n_docs = len(reasonings)
    avg_words = (total_words / n_docs) if n_docs else 0.0
    avg_chars = (total_chars / n_docs) if n_docs else 0.0
    avg_sents = (total_sents / n_docs) if n_docs else 0.0
    lex_div   = (total_unique / max(1, total_tokens))  # unique/total across corpus

    # rates
    hedge_rate_per100 = rate_per_100_words(hedge_hits, total_words)
    conf_rate_per100  = rate_per_100_words(conf_hits, total_words)
    moral_per1000     = rate_per_1000_words(moral_hits, total_words)
    relational_per1000= rate_per_1000_words(relational_hits, total_words)
    epistemic_per1000 = rate_per_1000_words(epistemic_hits, total_words)

    # sentiment aggregate
    avg_sentiment = sum(sentiment_scores)/n_docs if n_docs else 0.0

    return {
        "n_reasonings": n_docs,
        "avg_words": round(avg_words, 3),
        "avg_chars": round(avg_chars, 3),
        "avg_sentences": round(avg_sents, 3),
        "lexical_diversity": round(lex_div, 4),
        "hedge_rate_per100w": round(hedge_rate_per100, 3),
        "confidence_rate_per100w": round(conf_rate_per100, 3),
        "pct_with_justification_marker": round(100.0 * (justi_hits / n_docs), 2) if n_docs else 0.0,
        "moral_terms_per1000w": round(moral_per1000, 3),
        "relational_terms_per1000w": round(relational_per1000, 3),
        "epistemic_terms_per1000w": round(epistemic_per1000, 3),
        "avg_sentiment_lexicon": round(avg_sentiment, 3),
    }

# -----------------------------
# Loading helpers
# -----------------------------
def _robust_read_csv(path: str) -> Optional[pd.DataFrame]:
    encodings = ["utf-8", "utf-8-sig", "latin1"]
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    try:
        return pd.read_csv(path)  # let pandas guess
    except Exception as e:
        print(f"[WARN] Could not read {path}: {e}")
        return None

def find_reasoning_column(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    lowered = {c.lower(): c for c in df.columns}
    # exact candidates
    for cand in candidates:
        if cand.lower() in lowered:
            return lowered[cand.lower()]
    # fallback guesses
    for c in df.columns:
        cl = c.lower()
        if ("reason" in cl) or ("explain" in cl) or ("rational" in cl) or ("why" == cl):
            return c
    return None

def normalize_choice(val) -> Optional[str]:
    if isinstance(val, str):
        v = val.strip().upper()
    else:
        try:
            v = str(val).strip().upper()
        except Exception:
            return None
    if v in {"A", "B", "C"}:
        return v
    m = re.search(r"\b([ABC])\b", v)
    return m.group(1) if m else None

# -----------------------------
# Main analysis
# -----------------------------
def run_analysis(model_files: Dict[str, str],
                 choice_col: str = "choice",
                 reasoning_cols: List[str] = None,
                 output_dir: str = ".",
                 lexicons = DEFAULT_LEXICONS) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    model_files: mapping of model name -> CSV path
    choice_col: column containing option choice (A/B/C)
    reasoning_cols: candidate columns for reasoning/explanation text
    """
    if reasoning_cols is None:
        reasoning_cols = ["reasoning", "explanation", "rationale", "why"]

    records = []
    counts_records = []

    os.makedirs(output_dir, exist_ok=True)

    for model, path in model_files.items():
        if not os.path.exists(path):
            print(f"[WARN] File not found for {model}: {path} (skipping)")
            continue

        df = _robust_read_csv(path)
        if df is None:
            print(f"[WARN] Skipping unreadable file: {path}")
            continue

        # Choice column
        if choice_col not in df.columns:
            guess = None
            for c in df.columns:
                cl = c.lower()
                if ("choice" in cl) or ("option" in cl) or (cl == "answer") or ("selected" in cl):
                    guess = c
                    break
            if guess is None:
                print(f"[WARN] '{choice_col}' not in {path} and no guess found (skipping)")
                continue
            else:
                choice_col_eff = guess
        else:
            choice_col_eff = choice_col

        # Reasoning column
        reason_col = find_reasoning_column(df, list(reasoning_cols))
        if reason_col is None:
            print(f"[WARN] No reasoning column found in {path} (skipping)")
            continue

        # Normalize choices & drop NaNs
        df["_CHOICE_"] = df[choice_col_eff].apply(normalize_choice)
        df = df.dropna(subset=["_CHOICE_"])

        for opt in ["A", "B", "C"]:
            opt_reasonings = (
                df.loc[df["_CHOICE_"] == opt, reason_col]
                .fillna("")
                .astype(str)
                .tolist()
            )
            metrics = analyze_reasonings(opt_reasonings, lexicons)
            rec = {"model": model, "option": opt, **metrics}
            records.append(rec)

            counts_records.append({
                "model": model,
                "option": opt,
                "n_reasonings": metrics["n_reasonings"]
            })

    summary_df = pd.DataFrame(records)
    counts_df  = pd.DataFrame(counts_records)

    out1 = os.path.join(output_dir, "reasoning_metrics_by_model_option.csv")
    out2 = os.path.join(output_dir, "reasoning_counts_by_model_option.csv")
    summary_df.to_csv(out1, index=False)
    counts_df.to_csv(out2, index=False)

    print(f"[OK] Saved: {out1}")
    print(f"[OK] Saved: {out2}")
    return summary_df, counts_df

# -----------------------------
# Comparisons across LLMs by option (A/B/C)
# -----------------------------
DEFAULT_COMPARE_METRICS: Sequence[str] = (
    "n_reasonings",
    "avg_words",
    "avg_sentences",
    "lexical_diversity",
    "hedge_rate_per100w",
    "confidence_rate_per100w",
    "pct_with_justification_marker",
    "moral_terms_per1000w",
    "relational_terms_per1000w",
    "epistemic_terms_per1000w",
    "avg_sentiment_lexicon",
)

def compare_across_models(summary_df: pd.DataFrame,
                          option: str = "A",
                          metrics: Sequence[str] = DEFAULT_COMPARE_METRICS,
                          output_dir: str = ".") -> pd.DataFrame:
    """
    Create a comparison table for one option across models and save per-metric rankings.
    Returns the comparison dataframe for the option.
    """
    os.makedirs(output_dir, exist_ok=True)
    comp_dir = os.path.join(output_dir, "comparisons")
    os.makedirs(comp_dir, exist_ok=True)

    sub = summary_df[summary_df["option"] == option].copy()
    if sub.empty:
        print(f"[WARN] No rows for option {option}")
        return pd.DataFrame()

    cols = ["model", "option"] + [m for m in metrics if m in sub.columns]
    sub = sub[cols].copy()

    combined_path = os.path.join(comp_dir, f"option_{option}_comparison.csv")
    sub.to_csv(combined_path, index=False)

    for m in metrics:
        if m not in sub.columns:
            continue
        ranked = sub.sort_values(m, ascending=False).reset_index(drop=True)
        ranked.to_csv(os.path.join(comp_dir, f"rankings_option_{option}_{m}.csv"), index=False)

    print(f"[OK] Wrote comparisons for option {option} to: {comp_dir}")
    return sub

def compare_all_options(summary_df: pd.DataFrame,
                        metrics: Sequence[str] = DEFAULT_COMPARE_METRICS,
                        output_dir: str = ".") -> Dict[str, pd.DataFrame]:
    """Run compare_across_models for options A, B, and C; return dict of option -> df."""
    results = {}
    for opt in ["A","B","C"]:
        results[opt] = compare_across_models(summary_df, option=opt, metrics=metrics, output_dir=output_dir)
    return results

# -----------------------------
# Optional plotting helper (saves PNGs if --plots)
# -----------------------------
def save_option_metric_plot(summary_df: pd.DataFrame,
                            metric: str,
                            option: str,
                            out_dir: str):
    import matplotlib.pyplot as plt  # stdlib import; no seaborn, no explicit colors
    sub = summary_df[summary_df["option"] == option].copy()
    if sub.empty or metric not in sub.columns:
        return
    sub = sub.sort_values(metric, ascending=False)
    os.makedirs(out_dir, exist_ok=True)
    plt.figure()
    plt.bar(sub["model"], sub[metric])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel(metric)
    plt.title(f"{metric} — Option {option}")
    plt.tight_layout()
    out_path = os.path.join(out_dir, f"{metric}_option_{option}.png")
    plt.savefig(out_path, dpi=200)
    plt.close()

# -----------------------------
# CLI (robust to Jupyter's injected -f arg)
# -----------------------------
def parse_args(argv=None):
    ap = argparse.ArgumentParser(description="Analyze LLM reasoning lines by MODEL × OPTION and compare across LLMs per option.")
    ap.add_argument("--map", type=str, default=None,
                    help="Path to JSON file mapping model->csv_path.")
    ap.add_argument("--file", action="append", default=[],
                    help='Add a single mapping as "MODEL=/path/file.csv". Can be used multiple times.')
    ap.add_argument("--choice-col", type=str, default="choice",
                    help="Column containing the A/B/C choice (default: choice).")
    ap.add_argument("--reason-cols", type=str, default="reasoning,explanation,rationale,why",
                    help="Comma-separated candidate columns for reasoning text.")
    ap.add_argument("--output-dir", type=str, default=".",
                    help="Directory to write outputs (default: current dir).")
    ap.add_argument("--plots", action="store_true",
                    help="If set, save simple PNG plots for key metrics per option.")
    # Hidden arg to swallow Jupyter's injected -f flag
    ap.add_argument("-f", "--f", default=None, help=argparse.SUPPRESS)
    # Parse *known* args so any other injected args won't crash
    return ap.parse_known_args(argv)[0]

def build_model_map(args) -> Dict[str, str]:
    model_files = {}
    if args.map:
        with open(args.map, "r", encoding="utf-8") as f:
            data = json.load(f)
        if not isinstance(data, dict):
            raise ValueError("--map JSON must be an object of {model: path}")
        model_files.update(data)
    for pair in args.file:
        if "=" not in pair:
            raise ValueError(f"--file must be MODEL=PATH, got: {pair}")
        model, path = pair.split("=", 1)
        model_files[model.strip()] = path.strip()
    if not model_files:
        raise ValueError("No model files provided. Use --map or --file MODEL=PATH.")
    return model_files

def main(argv=None):
    args = parse_args(argv)
    model_files = build_model_map(args)
    reasoning_cols = [c.strip() for c in args.reason_cols.split(",") if c.strip()]

    summary_df, counts_df = run_analysis(
        model_files=model_files,
        choice_col=args.choice_col,
        reasoning_cols=reasoning_cols,
        output_dir=args.output_dir,
        lexicons=DEFAULT_LEXICONS
    )

    compare_all_options(summary_df, metrics=DEFAULT_COMPARE_METRICS, output_dir=args.output_dir)

    if args.plots:
        plots_dir = os.path.join(args.output_dir, "plots")
        metrics_to_plot = [
            "avg_words",
            "hedge_rate_per100w",
            "confidence_rate_per100w",
            "pct_with_justification_marker",
            "moral_terms_per1000w",
            "relational_terms_per1000w",
            "epistemic_terms_per1000w",
            "avg_sentiment_lexicon",
        ]
        for opt in ["A", "B", "C"]:
            for m in metrics_to_plot:
                save_option_metric_plot(summary_df, metric=m, option=opt, out_dir=plots_dir)

if __name__ == "__main__":
    # Pass sys.argv[1:] so parse_args can swallow unknown Jupyter args safely.
    try:
        main(sys.argv[1:])
    except Exception as e:
        print(f"[ERROR] {e}")
        print("Tip: In notebooks, call the functions directly instead of the CLI.\n"
              "Example:\n"
              "  from reasoning_analysis import run_analysis, compare_all_options\n"
              "  summary, counts = run_analysis({...}, choice_col='choice',\n"
              "                                  reasoning_cols=['reasoning','explanation'], output_dir='.')\n"
              "  compare_all_options(summary, output_dir='.')")


[ERROR] No model files provided. Use --map or --file MODEL=PATH.
Tip: In notebooks, call the functions directly instead of the CLI.
Example:
  from reasoning_analysis import run_analysis, compare_all_options
  summary, counts = run_analysis({...}, choice_col='choice',
                                  reasoning_cols=['reasoning','explanation'], output_dir='.')
  compare_all_options(summary, output_dir='.')
